# Copiloto WIN 2.0 — execução temporária na nuvem (gratuita)
**Abra pelo celular no Google Colab, com a sessão CPU padrão. Sem Oracle, sem cartão e sem notebook ligado.**

**Status real:** este laboratório instala e verifica Windows x64 no Linux/Wine. Só tentará autenticação Nelogica se você executar a última célula, depois de substituir a senha publicada anteriormente e confirmar que sua assinatura permite uso em nuvem. Não garante receber preços; ainda não conecta dados ao painel online.

**Privacidade:** sem senha/CPF/chave no GitHub ou no notebook salvo. As três credenciais da DLL serão solicitadas separadamente, em campos temporários, e não aparecem nos comandos. Não instale extensões desconhecidas. Não use senha antiga compartilhada no chat. Colab não é cofre de segredos permanente.

**Limites:** sessão gratuita do Colab é temporária (pode parar antes de 12 horas), não garante hardware nem atividade contínua. Use para TESTES interativos, não para alertas operacionais.

Fonte Nelogica: https://ajuda.nelogica.com.br/hc/pt-br/articles/54973527417243-ProfitDLL-no-Linux-Saiba-como-acessar-e-utilizar

In [ ]:
import os, pathlib, platform, subprocess, time, tempfile, urllib.request, zipfile, json
from pathlib import Path
ROOT=Path("/content/copiloto_win20")  # somente no runtime temporário
ROOT.mkdir(parents=True,exist_ok=True)
PREFIX=ROOT/"wine"
WIN_PY=ROOT/"python_win"/"python.exe"
SDK=ROOT/"sdk"
RUNTIME=ROOT/"dll_win64"
ENV=os.environ.copy()
ENV.update({"WINEPREFIX":str(PREFIX),"WINEARCH":"win64","WINEDEBUG":"-all","DEBIAN_FRONTEND":"noninteractive"})
print("Máquina:", platform.platform(),"| arquitetura:",platform.machine())
if platform.machine() != "x86_64": raise RuntimeError("Este caminho precisa de processador x86_64. Não instale Wine ARM para DLL Win64.")
print("Pasta temporária:",ROOT,"| Apenas validação, SEM TRADE.")


In [ ]:
import subprocess
# Instala pacotes públicos e configura um ambiente temporário sem credenciais.
subprocess.run(["apt-get","update","-qq"],check=True)
subprocess.run(["apt-get","install","-y","-qq","--no-install-recommends","wine","wine64","xvfb","unzip","wget","ca-certificates"],check=True)
print(subprocess.run(["wine","--version"],capture_output=True,text=True).stdout.strip())
PREFIX.mkdir(parents=True,exist_ok=True)
# wait wineserver with Xvfb still running (fix validated via GitHub Actions).
cmd='set -e; wineboot --init; timeout 90s wineserver --wait; test -f "$WINEPREFIX/system.reg"; grep -q "#arch=win64" "$WINEPREFIX/system.reg"'
subprocess.run(["timeout","180","xvfb-run","-a","bash","-c",cmd],check=True,env=ENV)
print("Wine x64 preparado. A ProfitDLL ainda não foi utilizada.")


In [ ]:
# Python nativo do Linux NÃO deve abrir ProfitDLL.dll.
# Precisamos de um python.exe Windows 64 bits executado pelo Wine.
URL="https://www.python.org/ftp/python/3.13.15/python-3.13.15-embed-amd64.zip"
ZIP=ROOT/"python_windows_embeddable.zip"
if not WIN_PY.exists():
    urllib.request.urlretrieve(URL,ZIP)
    with zipfile.ZipFile(ZIP) as z:
        corrupt=z.testzip()
        if corrupt: raise RuntimeError("Falha de integridade do Python Windows")
        WIN_PY.parent.mkdir(exist_ok=True)
        z.extractall(WIN_PY.parent)
script='import ctypes,sys,struct; assert sys.platform=="win32"; assert struct.calcsize("P")==8; assert ctypes.sizeof(ctypes.c_wchar)==2; assert hasattr(ctypes,"WinDLL"); assert hasattr(ctypes,"WINFUNCTYPE"); print("ABI WINDOWS 64 BITS OK")'
r=subprocess.run(["timeout","45","xvfb-run","-a","wine",str(WIN_PY),"-c",script],env=ENV,capture_output=True,text=True)
print(r.stdout[-1800:])
if r.returncode: raise RuntimeError("Falha de ABI Windows x64. Verifique o runtime; não prossiga.")
print("Pré-requisito Python/Wine VALIDADO. Ainda NÃO há conexão com a Nelogica.")


In [ ]:
# Baixa a distribuição oficial da Nelogica, nunca copia para um repositório público.
# Se a URL oficial exigir autenticação, esta etapa para sem tentar contornar o acesso.
URL="https://download-setup.nelogica.com.br/connector/latest/ProfitDLL.zip"
ARCHIVE=ROOT/"ProfitDLL.zip"
if not ARCHIVE.is_file():
    try:
        req=urllib.request.Request(URL,headers={"User-Agent":"CopilotoWIN2-Check/1.0"})
        with urllib.request.urlopen(req,timeout=60) as web,open(ARCHIVE,"wb") as out:
            while True:
                chunk=web.read(2**20)
                if not chunk: break
                out.write(chunk)
    except Exception as exc:
        ARCHIVE.unlink(missing_ok=True)
        raise RuntimeError("Download oficial não funcionou. Não envie credenciais ao notebook. Acesse sua área logada e confira o URL oficial.") from None
with zipfile.ZipFile(ARCHIVE) as z:
    if z.testzip(): raise RuntimeError("Arquivo ZIP da ProfitDLL corrompido")
    items=[n for n in z.namelist() if "/DLLs/Win64/" in "/"+n.replace("\\","/")]
    if not items: raise RuntimeError("Pasta DLLs/Win64 não encontrada no ZIP. Precisa conferir a versão do pacote.")
    RUNTIME.mkdir(parents=True,exist_ok=True)
    for name in items:
        norm=name.replace("\\","/")
        relative=norm.split("DLLs/Win64/",1)[1]
        if not relative or relative.startswith("/") or ".." in pathlib.PurePosixPath(relative).parts: continue
        dest=RUNTIME.joinpath(*pathlib.PurePosixPath(relative).parts)
        if name.endswith("/"):dest.mkdir(parents=True,exist_ok=True);continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(name) as src,open(dest,"wb") as target:
            import shutil; shutil.copyfileobj(src,target)
if not (RUNTIME/"ProfitDLL.dll").is_file():raise RuntimeError("ProfitDLL.dll ausente no runtime oficial.")
print("Runtime oficial Win64 extraído apenas no armazenamento temporário privado do Colab.")
print("DLL principal encontrada, outros arquivos preservados. Nenhuma conexão ainda.")


In [ ]:
# Checagem de carregamento e exports, SEM qualquer dado de conta.
def winepath(p):
    r=subprocess.run(["xvfb-run","-a","winepath","-w",str(p)],env=ENV,capture_output=True,text=True,timeout=25)
    if r.returncode:raise RuntimeError("Não foi possível obter caminho Wine.")
    return r.stdout.strip().splitlines()[-1]
WIN_RUNTIME=winepath(RUNTIME)
scan=r"""
import ctypes as C,os,sys,json
d=sys.argv[1]
try:
    os.chdir(d)
    if hasattr(os,"add_dll_directory"): os.add_dll_directory(d)
    dll=C.WinDLL(os.path.join(d,"ProfitDLL.dll"))
    names=["DLLInitializeMarketLogin","SetTradeCallbackV2","TranslateTrade","SubscribeTicker","UnsubscribeTicker","DLLFinalize"]
    missing=[name for name in names if not hasattr(dll,name)]
    print(json.dumps({"loaded":True,"market_data_exports_ok":not missing,"missing":missing,"order_routing_used":False}))
except Exception:
    print(json.dumps({"loaded":False,"note":"Falha ao carregar ou dependência Win64 ausente. Nenhum login executado."}))
    sys.exit(2)
"""
scan_path=ROOT/"scan_dll.py"
scan_path.write_text(scan,encoding="utf-8")
r=subprocess.run(["timeout","60","xvfb-run","-a","wine",str(WIN_PY),str(scan_path),WIN_RUNTIME],env=ENV,capture_output=True,text=True,timeout=80)
print(r.stdout[-2000:] if r.stdout else "Sem retorno de verificação.")
if r.returncode:print("Ainda NÃO passou no carregamento da DLL. Não tente autenticação.")
else:print("Carregamento da DLL passou. Ainda NÃO testamos licença nem cotação.")


In [ ]:
# OPCIONAL, somente após nova senha da Nelogica e aprovação de uso da sua licença em nuvem.
# Executa uma única sessão de HANDSHAKE SOMENTE MARKET DATA; não assina ativos nem envia ordens.
# NÃO clique em "Executar tudo": esta etapa pede confirmação de você antes de prosseguir.
from getpass import getpass
confirmacao=input("Sua licença permite teste em nuvem e você trocou a senha antiga? Digite SIM para prosseguir: ").strip().upper()
if confirmacao != "SIM": raise RuntimeError("Login cancelado pelo usuário. Nada foi enviado.")
if not (RUNTIME/"ProfitDLL.dll").exists(): raise RuntimeError("Primeiro execute e valide as células anteriores.")
chave=getpass("Chave de ativação DLL (oculta): ")
usuario=getpass("Login Nelogica (oculto): ")
senha=getpass("SENHA NOVA Nelogica (oculta): ")
if not all([chave,usuario,senha]): raise RuntimeError("Credenciais incompletas.")
import textwrap
source=r"""
import ctypes as C,json,os,sys,time
from threading import Lock
d=sys.argv[1];os.chdir(d)
if hasattr(os,"add_dll_directory"):os.add_dll_directory(d)
api=C.WinDLL(os.path.join(d,"ProfitDLL.dll"))
CB=C.WINFUNCTYPE(None,C.c_int,C.c_int)
status={};guard=Lock()
@CB
def onchange(kind,code):
    with guard:status[int(kind)]=int(code)
api.DLLInitializeMarketLogin.argtypes=[C.c_wchar_p,C.c_wchar_p,C.c_wchar_p,CB,*([C.c_void_p]*7)]
api.DLLInitializeMarketLogin.restype=C.c_int
api.DLLFinalize.argtypes=[];api.DLLFinalize.restype=C.c_int
def credential():return sys.stdin.readline().rstrip("\r\n")
a,b,c=credential(),credential(),credential()
started=False
try:
    result=api.DLLInitializeMarketLogin(a,b,c,onchange,None,None,None,None,None,None,None)
    a=b=c=""
    if result!=0:
        print(json.dumps({"handshake":"REJEITADO","return_code":result,"operational":"SEM TRADE"}));sys.exit(1)
    started=True
    deadline=time.monotonic()+45
    while time.monotonic()<deadline:
        with guard: s=dict(status)
        if s.get(0)==0 and s.get(2)==4 and s.get(3)==0:
            print(json.dumps({"handshake":"LOGIN_MARKET_ATIVACAO_OK","operational":"SEM TRADE","subscribed":False,"trades_received":0}));break
        time.sleep(.1)
    else:
        print(json.dumps({"handshake":"PENDENTE_TIMEOUT","states":s,"operational":"SEM TRADE"}))
finally:
    if started:
        try:api.DLLFinalize()
        except Exception:pass
"""
probe=ROOT/"cloud_market_handshake.py";probe.write_text(source,encoding="utf-8")
payload="\n".join([chave,usuario,senha])+"\n"
try:
    result=subprocess.run(["timeout","80","xvfb-run","-a","wine",str(WIN_PY),str(probe),WIN_RUNTIME],
       input=payload,text=True,capture_output=True,env=ENV,timeout=85)
finally:
    del payload,chave,usuario,senha
print("Código de saída:",result.returncode)
print("Resposta do handshake (sem credenciais):",result.stdout[-1200:].strip() or "Sem resposta. Verifique dependências e logs locais.")
print("Sem assinaturas de WIN, sem streaming de cotações e sem ordens. Encerramento automático.")
